In [1]:
import numpy as np

In [2]:
class Tensor:

    def __init__(self, data, parents=(), creator=None, req_grad=False):
        self.data = np.asarray(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self.parents = parents
        self.creator = creator
        self.requires_grad = req_grad

    def __repr__(self):
        op_name = "empty" if self.creator is None else f"{type(self.creator).__name__}"
        return f"Data={self.data}, grad={self.grad}, op={op_name}, requires_grad={self.requires_grad}"

    def __neg__(self):
        return Neg.run(self)
    @property
    def shape(self):
        return self.data.shape
    
    def backward(self):
        if self.requires_grad == False:
            raise RuntimeError("No trainable parameter found set req_grad=True on your parameters")
        topo_arr = []
        self.topo_sort(set(),set(), topo_arr)
        topo_arr.reverse()
        for i in range(len(topo_arr)):
            if topo_arr[i].creator is None:
                continue
            else:
                topo_arr[i].grad = np.zeros_like(topo_arr[i].data)
        self.grad = np.ones_like(self.data)
        for i in range(len(topo_arr)):
            if topo_arr[i].creator is None:
                continue
            if topo_arr[i].requires_grad == True:
                parent_gradients = topo_arr[i].creator.backward(topo_arr[i].grad)
                for n,s in zip(topo_arr[i].parents, parent_gradients):
                    if n.requires_grad == True:
                        if n.grad.shape != s.shape:
                            s = self.unbroadcast(n,s)
                        n.grad += s

    def unbroadcast(self, tgt_grad, act_grad):
        for i in range(len(act_grad.shape) - len(tgt_grad.shape)):
            act_grad = np.sum(act_grad, axis = 0)
        if len(tgt_grad.shape) != 0:
            for j in range(len(tgt_grad.shape)):
                if act_grad.shape[j] != tgt_grad.shape[j]:
                    act_grad = np.sum(act_grad, axis = j, keepdims=True)

        return act_grad    

    def zero_grad(self):
        self.grad = np.zeros_like(self.data)

    def topo_sort(self, visiting, visited, result):
        visiting.add(self)
        for i in range(len(self.parents)):
            if self.parents[i] not in visited:
                self.parents[i].topo_sort(visiting, visited, result)
        visiting.remove(self)
        visited.add(self)
        result.append(self)
        return


        



In [ ]:
import numpy as np


class operations:

    def forward(self, *args, **kwargs):
        raise NotImplementedError

    def backward(self, *args, **kwargs):
        raise NotImplementedError

    @classmethod
    def run(cls, *args, **kwargs):
        obj = cls()
        args = list(args)
        for i in range(len(args)):
            if not isinstance(args[i], Tensor):
                args[i] = Tensor(args[i])
                
        inputs = tuple(a.data for a in args)
        result = obj.forward(*inputs, **kwargs)
        obj.output = result
        for a in args:
            if a.requires_grad == True:
                req_grad = True
                break
            else:
                req_grad = False
        return Tensor(result, parents=args, creator=obj, req_grad=req_grad)


def grad_check(ops, *args):
    obj = ops()
    args = list(args)
    for i in range(len(args)):
        if not isinstance(args[i], Tensor):
            args[i] = Tensor(args[i])
    eps = Tensor(1e-6)
    def nudged(arr,idx,delta):
        out = arr.copy()
        out[idx] += delta
        return out
    for i in range(len(args)):
        if args[i].requires_grad == False:
            continue
        for idx in np.ndindex(args[i].data.shape):
            inputs = tuple(nudged(args[j].data, idx, eps.data) if j == i else args[j].data for j in range(len(args)))
            psv_res = obj.forward(*inputs)
            inputs = tuple(nudged(args[j].data, idx, -eps.data) if j == i else args[j].data for j in range(len(args)))
            neg_res = obj.forward(*inputs)
            grad = np.sum((psv_res - neg_res)/(2*eps.data))
            rel_error = abs(grad - args[i].grad[idx])/np.maximum(np.maximum(abs(grad),abs(args[i].grad[idx])),1e-12)
            if np.isnan(rel_error).any():
                raise RuntimeError("Gradient is undefined")
            if np.max(rel_error) >= 1e-5:
                return False
    return True

class Mul(operations):
    def forward(self, x, y):
        self.x,self.y = x,y
        return self.x * self.y
    def backward(self, gradient):
        return tuple([self.y*gradient, self.x*gradient])

class Add(operations):
    def forward(self, x, y):
        self.x,self.y = x,y
        return self.x + self.y
    def backward(self, gradient):
        return tuple([gradient, gradient])
    
class Sub(operations):
    def forward(self, x, y):
        self.x,self.y = x,y
        return self.x - self.y
    def backward(self, gradient):
        return tuple([gradient, -gradient])
    
class Div(operations):
    def forward(self, x, y):
        self.x,self.y = x,y
        return self.x * (self.y ** -1)
    def backward(self, gradient):
        return tuple([gradient*(self.y ** -1), gradient*self.x*(-self.y ** -2)])

class Pow(operations):
    def forward(self, x, y):
        self.x,self.y = x,y
        return self.x ** self.y
    def backward(self, gradient):
        return tuple([self.y * (self.x**(self.y - 1)) * gradient, (self.x ** self.y)*np.log(self.x)*gradient])

class Neg(operations):
    def forward(self, x):
        return -x
    def backward(self, gradient):
        return tuple([-gradient])

class Exp(operations):
    def forward(self, x):
        self.x = x
        return np.exp(self.x)
    def backward(self, gradient):
        return tuple([gradient*np.exp(self.x)])

class Log(operations):
    def forward(self, x):
        self.x = x
        return np.log(self.x)
    def backward(self, gradient):
        return tuple([gradient*(self.x**(-1))])

class ReLU(operations):
    def forward(self, x):
        self.x = x
        return np.maximum(self.x, 0)
    def backward(self, gradient):
        result = self.x
        for idx in np.ndindex(result):
            if result[idx] > 0:
                result[idx] = gradient[idx]
            else:
                result[idx] = np.float64(0)

In [3]:
import torch
import numpy as np
for v in [0., -2.]:
    t = torch.tensor(v, requires_grad=True)
    y = torch.log(t); y.backward()
    print(v, y.item(), t.grad.item())
a = np.array([1., -1., 0.])
print(a / 0)
print(np.log(np.array([0., -1.])))

0.0 -inf inf
-2.0 nan -0.5
[ inf -inf  nan]
[-inf  nan]


C:\Users\rahul\AppData\Local\Temp\ipykernel_9676\3781857838.py:8: RuntimeWarning: divide by zero encountered in divide
  print(a / 0)
C:\Users\rahul\AppData\Local\Temp\ipykernel_9676\3781857838.py:8: RuntimeWarning: invalid value encountered in divide
  print(a / 0)
C:\Users\rahul\AppData\Local\Temp\ipykernel_9676\3781857838.py:9: RuntimeWarning: divide by zero encountered in log
  print(np.log(np.array([0., -1.])))
C:\Users\rahul\AppData\Local\Temp\ipykernel_9676\3781857838.py:9: RuntimeWarning: invalid value encountered in log
  print(np.log(np.array([0., -1.])))


In [6]:
x = Tensor([1., 2.], req_grad=True)
print(-x)

Data=[-1. -2.], grad=[0. 0.], op=Neg, requires_grad=True


In [9]:
# Test 1: squared error, where the base is often negative
x = Tensor(np.array([-3., 2.]), req_grad=True)
z = Pow.run(x, 2)
z.backward()
print(x.grad)        # expected: [-6.  4.]

# Test 2: reciprocal (negative exponent, positive base)
x = Tensor(np.array([2., 4.]), req_grad=True)
z = Pow.run(x, -1)
z.backward()
print(x.grad)        # expected: [-0.25   -0.0625]



2.0
hi
[16.  6.]
-1.0
[0.5  0.75]


C:\Users\rahul\AppData\Local\Temp\ipykernel_32556\2566141703.py:91: RuntimeWarning: invalid value encountered in log
  return tuple([self.y * (self.x**self.y - 1) * gradient, (self.x ** self.y)*np.log(self.x)*gradient])


In [137]:
x = Tensor([1., 2.]); W = Tensor([3., 4.], req_grad=True)
c = Mul.run(x, W); c.backward()
print(x.grad, W.grad, c.requires_grad)   # want: zeros, [1. 2.], True

p = Tensor([1.]); q = Tensor([2.])
print(Mul.run(p, q).requires_grad)      # want False

[0. 0.] [1. 2.] True
False


In [138]:
# A: intermediate node with no grad
p = Tensor([1.]); q = Tensor([2.]); W = Tensor([3.], req_grad=True)
r = Mul.run(p, q)
c = Mul.run(r, W); c.backward()
print(p.grad, r.grad, W.grad)      # want [0.] ? [2.]  ← what should r.grad be?

[0.] [0.] [2.]


In [143]:
# B: nothing trainable at all
p = Tensor([1.]); q = Tensor([2.])
c = Mul.run(p, q); c.backward()
print(c.grad)

RuntimeError: No trainable parameter found

In [144]:
def test(name, got, want):
    ok = np.allclose(got, want)
    print(("PASS " if ok else "FAIL ") + name, "| got", got, "| want", want)

a = Tensor([2., 3.], req_grad=True)
c = Mul.run(a, 2); c.backward()
test("tensor*int", a.grad, [2., 2.])

a = Tensor([2., 3.], req_grad=True)
c = Mul.run(3, a); c.backward()
test("int*tensor", a.grad, [3., 3.])

a = Tensor([2., 3.], req_grad=True)
c = Mul.run(a, 0.5); c.backward()
test("tensor*float", a.grad, [0.5, 0.5])

a = Tensor([[1., 2.], [3., 4.]], req_grad=True)
c = Mul.run(a, np.array([10., 20.])); c.backward()
test("tensor*ndarray", a.grad, [[10., 20.], [10., 20.]])

a = Tensor([1., 1.], req_grad=True)
c = Mul.run(Mul.run(a, 2), 2); c.backward()
test("chained const", a.grad, [4., 4.])


PASS tensor*int | got [2. 2.] | want [2.0, 2.0]
PASS int*tensor | got [3. 3.] | want [3.0, 3.0]
PASS tensor*float | got [0.5 0.5] | want [0.5, 0.5]
PASS tensor*ndarray | got [[10. 20.]
 [10. 20.]] | want [[10.0, 20.0], [10.0, 20.0]]
PASS chained const | got [4. 4.] | want [4.0, 4.0]
